In [1]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd, numpy as np, json, os, xgboost as xgb
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from sklearn.metrics import accuracy_score, log_loss, brier_score_loss

BASE = '/content/drive/MyDrive/nba-xgboost'
DATA_DIR, MODEL_DIR = f"{BASE}/data", f"{BASE}/models"
os.makedirs(MODEL_DIR, exist_ok=True)

train = pd.read_csv(f"{DATA_DIR}/train.csv", parse_dates=['GAME_DATE'])
train = train.sort_values(['GAME_DATE','GAME_ID']).reset_index(drop=True)

DIFF = [c for c in train.columns if c.endswith('_DIFF')]
X, y = train[DIFF], train['HOME_WIN']

print(X.shape, "| xgboost version:", xgb.__version__)

Mounted at /content/drive
(8590, 19) | xgboost version: 3.4.1


In [2]:
tscv = TimeSeriesSplit(n_splits=5)

def cv_scores(model):
    accs, lls = [], []
    for tr, va in tscv.split(X):
        model.fit(X.iloc[tr], y.iloc[tr])
        p = model.predict_proba(X.iloc[va])[:, 1]
        accs.append(accuracy_score(y.iloc[va], p > 0.5))
        lls.append(log_loss(y.iloc[va], p))
    return np.mean(accs), np.mean(lls)

default = xgb.XGBClassifier(random_state=42, n_jobs=-1)
acc, ll = cv_scores(default)
print(f"Default model | CV accuracy {acc:.2%} | CV log-loss {ll:.4f}")

Default model | CV accuracy 57.48% | CV log-loss 0.7888


In [5]:
param_space = {
    'max_depth':        [2, 3, 4, 5, 6],
    'learning_rate':    [0.01, 0.02, 0.03, 0.05, 0.08, 0.1],
    'n_estimators':     [100, 200, 300, 500, 800],
    'subsample':        [0.6, 0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.5, 0.6, 0.7, 0.8, 1.0],
    'min_child_weight': [1, 3, 5, 10, 20],
    'gamma':            [0, 0.1, 0.3, 1],
    'reg_alpha':        [0, 0.1, 1, 5],
    'reg_lambda':       [1, 2, 5, 10],
}

In [6]:
search = RandomizedSearchCV(
    estimator=xgb.XGBClassifier(random_state=42, n_jobs=1),
    param_distributions=param_space,
    n_iter=60,
    cv=TimeSeriesSplit(n_splits=5),
    scoring={'acc': 'accuracy', 'logloss': 'neg_log_loss'},
    refit='logloss',
    n_jobs=-1,
    random_state=42,
    verbose=1,
)
search.fit(X, y)

print("Best params:", search.best_params_)
print(f"CV accuracy {search.cv_results_['mean_test_acc'][search.best_index_]:.2%}")
print(f"CV log-loss {-search.best_score_:.4f}")

Fitting 5 folds for each of 60 candidates, totalling 300 fits
Best params: {'subsample': 0.9, 'reg_lambda': 1, 'reg_alpha': 5, 'n_estimators': 200, 'min_child_weight': 20, 'max_depth': 2, 'learning_rate': 0.02, 'gamma': 0.1, 'colsample_bytree': 0.8}
CV accuracy 62.54%
CV log-loss 0.6478


In [7]:
best = search.best_params_.copy()
best.pop('n_estimators')

split = int(len(X) * 0.9)
X_fit, y_fit = X.iloc[:split], y.iloc[:split]
X_val, y_val = X.iloc[split:], y.iloc[split:]

es_model = xgb.XGBClassifier(
    **best, n_estimators=2000, early_stopping_rounds=50,
    eval_metric='logloss', random_state=42, n_jobs=-1)
es_model.fit(X_fit, y_fit, eval_set=[(X_val, y_val)], verbose=False)

n_trees = es_model.best_iteration + 1
print("Best number of trees:", n_trees)

Best number of trees: 293


In [8]:
final_params = {**best, 'n_estimators': n_trees}
final_model = xgb.XGBClassifier(**final_params, random_state=42, n_jobs=-1)
final_model.fit(X, y)
print("Final params:", final_params)

Final params: {'subsample': 0.9, 'reg_lambda': 1, 'reg_alpha': 5, 'min_child_weight': 20, 'max_depth': 2, 'learning_rate': 0.02, 'gamma': 0.1, 'colsample_bytree': 0.8, 'n_estimators': 293}


In [9]:
train_acc = accuracy_score(y, final_model.predict(X))
train_brier = brier_score_loss(y, final_model.predict_proba(X)[:, 1])
print(f"Train accuracy {train_acc:.2%} | Train Brier {train_brier:.4f}")

Train accuracy 64.17% | Train Brier 0.2218


In [10]:
final_model.save_model(f"{MODEL_DIR}/xgb_model.json")

with open(f"{MODEL_DIR}/feature_list.json", "w") as f:
    json.dump(DIFF, f)

with open(f"{MODEL_DIR}/best_params.json", "w") as f:
    json.dump({**final_params, 'xgboost_version': xgb.__version__}, f, indent=2)

print(os.listdir(MODEL_DIR))

['xgb_model.json', 'feature_list.json', 'best_params.json']


In [11]:
import xgboost as xgb, pandas as pd, json
BASE = '/content/drive/MyDrive/nba-xgboost'

m = xgb.XGBClassifier()
m.load_model(f"{BASE}/models/xgb_model.json")
feats = json.load(open(f"{BASE}/models/feature_list.json"))

train = pd.read_csv(f"{BASE}/data/train.csv")
print(m.predict_proba(train[feats].head(3))[:, 1])

[0.4760878 0.7439309 0.4953603]
